In [1]:
import pandas as pd
import re

df = pd.read_csv('datasets/csic_database.csv')

print("Loaded successfully.")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nclassification counts:")
print(f"  Normal    (0): {(df['classification']==0).sum():,}")
print(f"  Anomalous (1): {(df['classification']==1).sum():,}")
print(f"\nMethod counts:")
print(df['Method'].value_counts())

Loaded successfully.
Shape: 61,065 rows x 17 columns

classification counts:
  Normal    (0): 36,000
  Anomalous (1): 25,065

Method counts:
Method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64


In [2]:
df_normal = df[df['classification'] == 0].copy()
df_attack = df[df['classification'] == 1].copy()

print(f"Normal rows:    {len(df_normal):,}")
print(f"Attack rows:    {len(df_attack):,}")
print(f"Total:          {len(df_normal) + len(df_attack):,}")

print(f"\nNormal - Method breakdown:")
print(df_normal['Method'].value_counts())

print(f"\nAttack - Method breakdown:")
print(df_attack['Method'].value_counts())

Normal rows:    36,000
Attack rows:    25,065
Total:          61,065

Normal - Method breakdown:
Method
GET     28000
POST     8000
Name: count, dtype: int64

Attack - Method breakdown:
Method
GET     15088
POST     9580
PUT       397
Name: count, dtype: int64


In [3]:
def clean_url(url):
    url_str = str(url)
    clean = re.sub(r'\s+HTTP/\d\.\d\s*$', '', url_str).strip()
    return clean

df['url_clean']      = df['URL'].apply(clean_url)
df['request_length'] = df['url_clean'].str.len()

print("FEATURE: request_length")
print(f"  Normal  avg: {df[df['classification']==0]['request_length'].mean():.1f} chars")
print(f"  Attack  avg: {df[df['classification']==1]['request_length'].mean():.1f} chars")
print(f"\n  Normal  max: {df[df['classification']==0]['request_length'].max()}")
print(f"  Attack  max: {df[df['classification']==1]['request_length'].max()}")

print(f"\nSample cleaned URLs:")
for url in df['url_clean'].head(3):
    print(f"  {url[:80]}")

FEATURE: request_length
  Normal  avg: 70.0 chars
  Attack  avg: 97.6 chars

  Normal  max: 358
  Attack  max: 886

Sample cleaned URLs:
  http://localhost:8080/tienda1/index.jsp
  http://localhost:8080/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=1
  http://localhost:8080/tienda1/publico/anadir.jsp


In [4]:
method_map = {'GET': 0, 'POST': 1, 'PUT': 2}

df['method_encoded'] = df['Method'].map(method_map).fillna(3).astype(int)

print("FEATURE: method_encoded")
print("Mapping: GET=0, POST=1, PUT=2")
print()
print(df[['Method', 'method_encoded']].drop_duplicates().sort_values('method_encoded'))
print()
print("Distribution in Normal rows (classification=0):")
print(df[df['classification']==0]['method_encoded'].value_counts())
print()
print("Distribution in Attack rows (classification=1):")
print(df[df['classification']==1]['method_encoded'].value_counts())

FEATURE: method_encoded
Mapping: GET=0, POST=1, PUT=2

      Method  method_encoded
0        GET               0
2       POST               1
36024    PUT               2

Distribution in Normal rows (classification=0):
method_encoded
0    28000
1     8000
Name: count, dtype: int64

Distribution in Attack rows (classification=1):
method_encoded
0    15088
1     9580
2      397
Name: count, dtype: int64


In [5]:
def count_params(url):
    url_str = str(url)
    if '?' in url_str:
        query_part = url_str.split('?')[1]
        query_part = re.sub(r'\s+HTTP/\d\.\d\s*$', '', query_part)
        return len(query_part.split('&'))
    return 0

df['num_parameters'] = df['url_clean'].apply(count_params)

print("FEATURE: num_parameters")
print(f"  Normal  avg: {df[df['classification']==0]['num_parameters'].mean():.2f} params")
print(f"  Attack  avg: {df[df['classification']==1]['num_parameters'].mean():.2f} params")
print()
print("Normal - parameter count distribution:")
print(df[df['classification']==0]['num_parameters'].value_counts().sort_index().head(6))
print()
print("Attack - parameter count distribution:")
print(df[df['classification']==1]['num_parameters'].value_counts().sort_index().head(6))

FEATURE: num_parameters
  Normal  avg: 1.17 params
  Attack  avg: 2.23 params

Normal - parameter count distribution:
num_parameters
0     28000
1      3000
3      1000
5      2000
13     2000
Name: count, dtype: int64

Attack - parameter count distribution:
num_parameters
0     15456
1      2799
3      1343
5      2741
13     2726
Name: count, dtype: int64


In [6]:
df['body_length'] = df['content'].fillna('').astype(str).str.len()

print("FEATURE: body_length")
print(f"  Normal  avg: {df[df['classification']==0]['body_length'].mean():.1f} chars")
print(f"  Attack  avg: {df[df['classification']==1]['body_length'].mean():.1f} chars")
print(f"\n  Normal  max: {df[df['classification']==0]['body_length'].max()}")
print(f"  Attack  max: {df[df['classification']==1]['body_length'].max()}")

print("\nSample attack bodies that are long:")
long_attack = df[(df['classification']==1) & (df['body_length'] > 100)]
for _, row in long_attack.head(3).iterrows():
    print(f"\n  Length {row['body_length']}: {str(row['content'])[:100]}")

FEATURE: body_length
  Normal  avg: 20.3 chars
  Attack  avg: 48.6 chars

  Normal  max: 307
  Attack  max: 836

Sample attack bodies that are long:

  Length 146: id=2&nombre=Jam%F3n+Ib%E9rico&precio=85&cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+*+FROM+datos+W

  Length 118: modo=entrar&login=bob%40%3CSCRipt%3Ealert%28Paros%29%3C%2FscrIPT%3E.parosproxy.org&pwd=84m3ri156&rem

  Length 106: modo=insertar&precio=183%27%2C%270%27%2C%270%27%29%3Bwaitfor+delay+%270%3A0%3A15%27%3B--&B1=Pasar+po


In [7]:
FEATURES = ['request_length', 'method_encoded', 'num_parameters', 'body_length']

X = df[FEATURES]
y = df['classification']

print("=== FINAL FEATURE MATRIX ===")
print(f"X shape: {X.shape}  (rows, features)")
print(f"y shape: {y.shape}")
print()
print("First 5 rows of X:")
print(X.head())
print()
print("First 5 values of y (0=Normal, 1=Attack):")
print(y.head().tolist())
print()
print("=== NULL CHECK ON FEATURES ===")
print(X.isnull().sum())
print()
print("=== FEATURE STATS: Normal vs Attack ===")
for feat in FEATURES:
    n_mean = X[y==0][feat].mean()
    a_mean = X[y==1][feat].mean()
    diff   = ((a_mean - n_mean) / n_mean * 100) if n_mean > 0 else 0
    print(f"  {feat:20s}  Normal avg={n_mean:7.2f}  Attack avg={a_mean:7.2f}  (+{diff:.0f}%)")

=== FINAL FEATURE MATRIX ===
X shape: (61065, 4)  (rows, features)
y shape: (61065,)

First 5 rows of X:
   request_length  method_encoded  num_parameters  body_length
0              39               0               0            0
1             117               0               5            0
2              48               1               0           68
3             116               0               5            0
4              52               1               0           63

First 5 values of y (0=Normal, 1=Attack):
[0, 0, 0, 0, 0]

=== NULL CHECK ON FEATURES ===
request_length    0
method_encoded    0
num_parameters    0
body_length       0
dtype: int64

=== FEATURE STATS: Normal vs Attack ===
  request_length        Normal avg=  69.99  Attack avg=  97.61  (+39%)
  method_encoded        Normal avg=   0.22  Attack avg=   0.41  (+86%)
  num_parameters        Normal avg=   1.17  Attack avg=   2.23  (+91%)
  body_length           Normal avg=  20.35  Attack avg=  48.58  (+139%)


In [8]:
X_normal = X[y == 0].copy()
X_attack = X[y == 1].copy()

print("=== TRAINING DATA (normal requests only) ===")
print(f"X_normal shape: {X_normal.shape}")
print(f"  This is what the Isolation Forest will TRAIN on")
print()
print("=== TEST DATA ===")
print(f"X_attack shape: {X_attack.shape}")
print(f"  This is what we test against to see if model catches attacks")
print()
print("Why train on normal only?")
print("  Isolation Forest is UNSUPERVISED — it learns what normal looks like")
print("  Then anything that doesn't look normal = anomaly flagged")
print("  We never show it attack examples during training")

=== TRAINING DATA (normal requests only) ===
X_normal shape: (36000, 4)
  This is what the Isolation Forest will TRAIN on

=== TEST DATA ===
X_attack shape: (25065, 4)
  This is what we test against to see if model catches attacks

Why train on normal only?
  Isolation Forest is UNSUPERVISED — it learns what normal looks like
  Then anything that doesn't look normal = anomaly flagged
  We never show it attack examples during training


In [9]:
X_normal.to_csv('datasets/X_normal_train.csv', index=False)
X.to_csv('datasets/X_all_features.csv', index=False)
y.to_csv('datasets/y_labels.csv', index=False)

print("Saved 3 files to datasets/ folder:")
print(f"  X_normal_train.csv  — {len(X_normal):,} rows, 4 cols (training data for Day 13)")
print(f"  X_all_features.csv  — {len(X):,} rows, 4 cols (all features for testing)")
print(f"  y_labels.csv        — {len(y):,} rows (0/1 labels for evaluation)")

print("\nVerify files saved:")
import os
for fname in ['X_normal_train.csv', 'X_all_features.csv', 'y_labels.csv']:
    path = f'datasets/{fname}'
    size = os.path.getsize(path) / 1024
    print(f"  {fname:30s}  {size:.1f} KB")

Saved 3 files to datasets/ folder:
  X_normal_train.csv  — 36,000 rows, 4 cols (training data for Day 13)
  X_all_features.csv  — 61,065 rows, 4 cols (all features for testing)
  y_labels.csv        — 61,065 rows (0/1 labels for evaluation)

Verify files saved:
  X_normal_train.csv              366.3 KB
  X_all_features.csv              632.9 KB
  y_labels.csv                    178.9 KB


## Day 12 Summary — CSIC 2010 Preprocessing

### 4 Features selected (from document)
| Feature        | Source column | How computed              | Normal avg | Attack avg |
|----------------|---------------|--------------------------|------------|------------|
| request_length | URL           | len(url after strip HTTP)| 70.0       | 97.6       |
| method_encoded | Method        | GET=0, POST=1, PUT=2     | 0.22       | 0.41       |
| num_parameters | URL           | count of & in query part | 1.17       | 2.23       |
| body_length    | content       | len(body or 0 if null)   | 20.35      | 48.58      |

### Files saved to datasets/
- X_normal_train.csv  — 36,000 rows — used for TRAINING on Day 13
- X_all_features.csv  — 61,065 rows — used for TESTING on Day 14
- y_labels.csv        — 61,065 rows — 0/1 ground truth labels

### Key insight
All 4 features are measurably higher in attacks than normal.
This gives the Isolation Forest a clear signal to learn from.